# MINA Voice API para Google Colab

Este notebook levanta una API para generar la voz clonada de MINA con GPU en Google Colab.

Uso recomendado:

1. En Colab activa GPU: `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`.
2. Ejecuta la celda `Setup fijo` una vez. Esa celda reinicia el runtime al final.
3. Ejecuta la celda `Levantar API`.
4. Sube `assets/voice/user_reference.wav` cuando Colab lo pida.
5. Copia el `VOICE_REMOTE_URL=...` que salga al final y ponlo en el `.env` de MINA.

Nota: las URLs gratuitas de Cloudflare/Colab cambian cuando se reinicia la sesión.


In [ ]:
# Setup fijo: instala dependencias compatibles y reinicia el runtime.
# Ejecuta esta celda una vez por sesión nueva de Colab.
import os
import subprocess
import sys
import time

print('Instalando Chatterbox y dependencias compatibles...')
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'chatterbox-tts',
    'fastapi',
    'uvicorn',
    'nest-asyncio',
    'python-multipart',
])
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--force-reinstall',
    'numpy==1.26.4',
    'scipy',
    'scikit-learn',
])
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'])

print('Setup listo. Reiniciando kernel limpio...')
time.sleep(1)
os.kill(os.getpid(), 9)


In [ ]:
# Levantar API de voz y abrir túnel público.
# Ejecuta esta celda después del reinicio provocado por el setup.
import os
import re
import subprocess
import tempfile
import threading
import time
import urllib.request
from pathlib import Path

cloudflared_path = Path('/content/cloudflared')
if not cloudflared_path.exists():
    print('Descargando cloudflared...')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cloudflared_path,
    )
    os.chmod(cloudflared_path, 0o755)

reference_path = Path('/content/user_reference.wav')
if not reference_path.exists():
    print('Sube el archivo assets/voice/user_reference.wav del proyecto de MINA.')
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No subiste ningún archivo de referencia.')
    first_file = next(iter(uploaded.keys()))
    Path(first_file).replace(reference_path)
print('Referencia lista:', reference_path, 'bytes:', reference_path.stat().st_size)

import nest_asyncio
import numpy
import torch
import torchaudio as ta
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel

print('NumPy:', numpy.__version__)

try:
    import perth

    if getattr(perth, 'PerthImplicitWatermarker', None) is None:
        class DummyWatermarker:
            def apply_watermark(self, wav, sample_rate=None):
                return wav

        perth.PerthImplicitWatermarker = DummyWatermarker
except Exception:
    pass

try:
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS
except Exception as error:
    print('Multilingual no disponible:', repr(error))
    ChatterboxMultilingualTTS = None
from chatterbox.tts import ChatterboxTTS

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_MULTILINGUAL = ChatterboxMultilingualTTS is not None
print('GPU/CPU:', DEVICE)

if USE_MULTILINGUAL:
    model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)
    model_name = 'ChatterboxMultilingualTTS'
else:
    model = ChatterboxTTS.from_pretrained(device=DEVICE)
    model.prepare_conditionals(str(reference_path), exaggeration=0.75)
    model_name = 'ChatterboxTTS'
print('Modelo listo:', model_name, 'sample_rate:', model.sr)

VOICE_TOKEN = os.environ.get('MINA_VOICE_TOKEN', '').strip()
app = FastAPI(title='MINA Voice API')

class GenerateRequest(BaseModel):
    text: str

def check_auth(authorization: str | None):
    if not VOICE_TOKEN:
        return
    if authorization != f'Bearer {VOICE_TOKEN}':
        raise HTTPException(status_code=401, detail='Token inválido')

@app.get('/health')
def health(authorization: str | None = Header(default=None)):
    check_auth(authorization)
    return {'ok': True, 'device': DEVICE, 'model': model_name, 'sample_rate': model.sr}

@app.post('/generate')
def generate(req: GenerateRequest, authorization: str | None = Header(default=None)):
    check_auth(authorization)
    text = req.text.strip()[:260]
    if not text:
        raise HTTPException(status_code=400, detail='Texto vacío')
    out_path = Path(tempfile.gettempdir()) / f'mina_voice_{abs(hash(text))}_{os.getpid()}.wav'
    if USE_MULTILINGUAL:
        wav = model.generate(
            text,
            'es',
            audio_prompt_path=str(reference_path),
            repetition_penalty=1.18,
            min_p=0.02,
            top_p=0.92,
            exaggeration=0.95,
            cfg_weight=0.60,
            temperature=0.96,
        )
    else:
        wav = model.generate(
            text,
            repetition_penalty=1.12,
            min_p=0.03,
            top_p=0.94,
            exaggeration=0.88,
            cfg_weight=0.48,
            temperature=0.90,
        )
    ta.save(str(out_path), wav, model.sr)
    return FileResponse(str(out_path), media_type='audio/wav', filename='mina.wav')

nest_asyncio.apply()
server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=7860, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
print('API local lista en http://127.0.0.1:7860')

proc = subprocess.Popen(
    [str(cloudflared_path), 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
logs = []
for _ in range(80):
    line = proc.stdout.readline()
    if line:
        logs.append(line.rstrip())
        print(line, end='')
        match = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(1)
if not public_url:
    print('\n'.join(logs[-30:]))
    raise RuntimeError('No se pudo obtener la URL pública de cloudflared.')
print('MINA_REMOTE_READY=' + public_url)
print('VOICE_REMOTE_URL=' + public_url)
